# LLM run inspection

Run one rendered scenario transition through an LLM, log the result, and inspect the submitted move.

Set at least
* `SCENARIO_NAME` to the name of a file in `outputs/scenarios`(without .json suffix!)
* `TRANSITION_INDEX` to the explicit transition from the selected scenario
* `MODEL_NAME` to the name of the model (options specified in `config/model_configs.yaml`)


The move is shown in one 2D plot per axis pair containing the move axis. A 4D move on axis 0 therefore produces the planes (0, 1), (0, 2), and (0, 3).

In [2]:
from IPython.display import display

from visualization.run_figures import (
    call_prepared_llm_transition,
    display_llm_prompt,
    display_llm_response,
    display_llm_run_summary,
    finalize_llm_transition,
    plot_llm_run_move,
    prepare_llm_transition,
)

In [3]:
SCENARIO_NAME = "evaluation_base"
TRANSITION_INDEX = 348
MODEL_NAME = "or_gpt-5-5"
REASONING_EFFORT = "high"

In [4]:
# Local preparation only: load scenario, reconstruct board, and build prompts.
prepared = prepare_llm_transition(
    scenario_name=SCENARIO_NAME,
    transition_index=TRANSITION_INDEX,
    model_name=MODEL_NAME,
    reasoning_effort=REASONING_EFFORT,
)
display_llm_prompt(prepared)


### Prompt sources

- System template: [`prompts/system.txt`](../prompts/system.txt)
- User template: [`prompts/user.txt`](../prompts/user.txt)
- Model config: [`config/model_configs.yaml`](../config/model_configs.yaml)
- Scenario: [`evaluation_base.json`](../outputs/scenarios/evaluation_base.json)
- Backend: `litellm`

### System prompt

```text
You solve a multidimensional formal-language Scrabble benchmark.
Follow the supplied rules exactly and return only one JSON object with exactly the fields `start`, `axis`, and `sequence`.
Do not use tools or add explanations.
```

### User prompt

```text
Place exactly one contiguous sequence. Maximize the number of newly placed rack symbols.

## Move geometry
- Coordinates are zero-based vectors with 2 entries.
- `start` is the first coordinate. `axis` advances one coordinate component per symbol.
- Axis selects the board dimension along which the sequence advances: axis 0 advances coordinate index 0, axis 1 advances coordinate index 1.

## Validity rules
- The submitted sequence must be accepted by the formal language.
- Existing cells may be reused only with their existing symbol.
- Reuse at least one existing cell and place at least one new symbol.
- Only newly placed symbols consume the rack, including multiplicities.
- Do not reuse a cell whose existing word already runs along the chosen axis.
- The cell immediately before and after the submitted sequence on its axis must not continue an existing word.
- A newly placed cell must not extend an already-valid word on any perpendicular axis.
- After placement, every maximal contiguous line of length greater than one that touches the move, on every axis, must be accepted by the formal language.

## Scoring
- Each alphabet symbol has a point value, listed below. A valid move scores the sum of the point values of every symbol in its placed sequence, counting both newly placed and reused (overlapping) symbols. Higher-value symbols and longer valid sequences score more.
- An invalid move scores 0.

Letter scores:
  A: 2
  D: 1
  E: 2
  H: 3
  I: 1
  U: 1
  V: 1
  Y: 2

Formal language:
Language ID: evaluation_base_grammar
Alphabet: {A, D, E, H, I, U, V, Y}
k: 3
Minimum word length: 3
Forbidden snippets: {A A D, A A E, A A H, A A Y, A D D, A D E, A D I, A D U, A E A, A E H, A E V, A E Y, A H D, A H E, A H H, A U D, A U E, A U H, A U I, A U U, A U V, A U Y, A V H, A V I, A Y A, A Y E, D A E, D A H, D A I, D A U, D A V, D D D, D D V, D D Y, D E D, D E U, D H A, D H D, D H E, D H V, D I I, D U A, D U E, D U Y, D V D, D V I, D V Y, D Y I, D Y U, D Y V, E A E, E A H, E A V, E D A, E D H, E D V, E E D, E E I, E E U, E H E, E H H, E H I, E H V, E H Y, E I A, E I E, E I U, E I V, E U A, E U D, E U Y, E V A, E V Y, E Y D, E Y H, E Y I, E Y Y, H A A, H A U, H D H, H D U, H D Y, H E E, H H E, H H H, H I A, H I I, H I V, H I Y, H U E, H U H, H U U, H V I, H V V, H Y D, H Y E, H Y I, I A A, I A D, I A E, I A Y, I D U, I D Y, I E E, I E H, I E I, I E V, I H D, I H H, I H Y, I I D, I I E, I I U, I U D, I U E, I V A, I V D, I V E, I V Y, I Y A, I Y D, I Y V, U A I, U A V, U D H, U D U, U E A, U E H, U E U, U E V, U E Y, U H A, U H H, U I D, U I H, U I U, U I V, U U D, U U V, U V A, U V Y, U Y D, U Y E, U Y U, U Y V, V A D, V A E, V D A, V D E, V D H, V E D, V E U, V E V, V H D, V H U, V H V, V H Y, V I A, V I H, V I V, V U D, V U H, V U V, V V A, V V H, V V I, Y A U, Y A V, Y D H, Y D I, Y D U, Y D V, Y D Y, Y E D, Y E E, Y E U, Y H D, Y I D, Y U E, Y U H, Y U U, Y U V, Y V D, Y V H, Y V U, Y Y A, Y Y E, Y Y I, Y Y V, Y Y Y}
A sequence is valid iff it has the minimum length of 3 and contains no forbidden snippet.

Board configuration:
[omitted from notebook display: 1342 occupied cells]

Rack:
["D", "D", "E", "H", "I", "Y"]

```


In [5]:
timed_response = call_prepared_llm_transition(prepared)

In [9]:
context = finalize_llm_transition(prepared, timed_response)
display_llm_run_summary(context)

sequence,OK
spatial,OK
overlap,OK
no word extension,OK
cross words,OK
rack,OK
word length,7
overlap count,1
letter score,13
prompt,36241
reasoning,2588


In [7]:
for figure in plot_llm_run_move(context, move_source="parsed"):
    display(figure)

# Keep in Mind

Ground truth != only solution

In [8]:
for figure in plot_llm_run_move(context, move_source="ground_truth"):
    display(figure)